<a href="https://colab.research.google.com/github/zhaocaiQ/project/blob/master/%EB%82%A0%EC%94%A8api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install requests


In [4]:
import requests

def get_weather(city_name, api_key):
    """
    OpenWeatherMap API를 이용해 현재 날씨 데이터를 가져오는 함수

    :param city_name: 조회할 도시 이름 (예: "Seoul")
    :param api_key: OpenWeatherMap API 키
    :return: 날씨 정보(JSON)
    """
    base_url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": api_key,
        "units": "metric",  # 섭씨 단위
        "lang": "kr"        # 한국어
    }

    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# 사용 예시
if __name__ == "__main__":
    api_key = "YOUR_API_KEY"  # 본인의 OpenWeatherMap API 키로 교체
    city = "Seoul"
    api_key = ""
    weather_data = get_weather(city, api_key)

    if weather_data:
        print("도시:", weather_data["name"])
        print("날씨:", weather_data["weather"][0]["description"])
        print("기온:", weather_data["main"]["temp"], "°C")
        print("체감 온도:", weather_data["main"]["feels_like"], "°C")
        print("풍속:", weather_data["wind"]["speed"], "m/s")


도시: Seoul
날씨: 맑음
기온: 16.76 °C
체감 온도: 15.51 °C
풍속: 6.17 m/s


| 온도      | 표현 예시                   |
| ------- | ----------------------- |
| < 0°C   | “매서운 찬바람이 뺨을 스친다.”      |
| 0–10°C  | “차가운 공기가 폐 속으로 스며든다.”   |
| 10–20°C | “선선한 아침 공기가 기분 좋게 감돈다.” |
| 20–28°C | “따사로운 햇살이 포근하게 느껴진다.”   |
| > 28°C  | “무더운 공기가 숨막히게 덮쳐온다.”    |


아침 6–9시: “이른 아침”, “햇살이 퍼지기 시작하는 시간”

오후 12–15시: “햇볕이 내리쬐는 대낮”

저녁 18–21시: “노을이 물드는 저녁”

밤 22시 이후: “깊은 밤의 정적 속에서”

In [5]:
import requests
from datetime import datetime, timedelta
import pytz

# 🧠 감성 문장 매핑
def describe_weather(weather_desc):
    mapping = {
        "clear sky": "햇살이 투명하게 내려앉은 맑은 하늘",
        "few clouds": "구름이 살짝 스친 고요한 하늘",
        "scattered clouds": "드문드문 구름이 흐르는 흐릿한 하늘",
        "broken clouds": "구름이 듬성듬성 떠다니는 흐린 하늘",
        "overcast clouds": "먹구름이 하늘을 덮은 잿빛 풍경",
        "light rain": "가벼운 빗방울이 속삭이듯 떨어진다",
        "moderate rain": "적당한 비가 조용히 세상을 적신다",
        "heavy rain": "굵은 빗줄기가 가슴 깊이 파고든다",
        "snow": "하얀 눈이 조용히 세상을 덮는다",
        "mist": "희뿌연 안개가 시야를 가린다"
    }
    return mapping.get(weather_desc, f"{weather_desc}의 날씨")

def temp_feeling(feels_like):
    if feels_like < 0:
        return "차가운 공기가 가슴 속까지 스며든다"
    elif feels_like < 10:
        return "서늘한 바람이 옷깃을 스친다"
    elif feels_like < 20:
        return "선선한 공기가 귓가를 감돈다"
    elif feels_like < 28:
        return "따스한 온기가 마음을 감싼다"
    else:
        return "무더운 공기가 피부에 들러붙는다"

def time_descriptor(hour):
    if 5 <= hour < 9:
        return "햇살이 막 떠오르는 이른 아침"
    elif 9 <= hour < 12:
        return "햇살이 퍼지는 아침"
    elif 12 <= hour < 17:
        return "햇볕이 내리쬐는 오후"
    elif 17 <= hour < 20:
        return "노을이 퍼지는 저녁"
    elif 20 <= hour < 24:
        return "조용한 밤의 공기"
    else:
        return "어둠이 깊게 내린 새벽"

# 🌤️ 날씨 API 요청
def get_weather_story(city_name, api_key):
    url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": api_key,
        "units": "metric",
        "lang": "en"
    }
    res = requests.get(url, params=params)
    if res.status_code != 200:
        return f"API 오류: {res.status_code}"

    data = res.json()

    # 한국 시간 변환
    utc_time = datetime.utcfromtimestamp(data['dt'])
    kst = pytz.timezone('Asia/Seoul')
    local_time = utc_time.replace(tzinfo=pytz.utc).astimezone(kst)
    hour = local_time.hour

    # 데이터 추출
    weather_desc = data["weather"][0]["description"]
    feels_like = data["main"]["feels_like"]

    # 감성 스토리 구성
    story = f"""{local_time.strftime('%Y년 %m월 %d일')} {time_descriptor(hour)}, {city_name}의 하늘은 {describe_weather(weather_desc)}.
그리고 {temp_feeling(feels_like)}."""

    return story

# 🧪 사용 예시
if __name__ == "__main__":
    api_key = ""
    city = "Seoul"
    story = get_weather_story(city, api_key)
    print("🎶 날씨 감성 스토리")
    print(story)


🎶 날씨 감성 스토리
2025년 05월 02일 햇볕이 내리쬐는 오후, Seoul의 하늘은 햇살이 투명하게 내려앉은 맑은 하늘.
그리고 선선한 공기가 귓가를 감돈다.


# 다양한 텍스트로 랜덤 선택

In [6]:
weather_mapping = {
    "clear sky": [
        "햇살이 유리창에 부서져서 내 방까지 번졌다",
        "맑은 하늘이 오늘만큼은 나를 응원하는 것 같아",
        "햇빛이 내 그림자 옆에 조용히 앉아 있었다",
        "푸른 하늘 아래 기타 코드가 맑게 울린다",
        "햇살이 눈을 감아도 느껴지는 날"
    ],
    "few clouds": [
        "하늘이 조금은 여유 있어 보였어, 나처럼",
        "조금의 구름은 어쩐지 혼잣말 같았고",
        "투명한 공기 속에 구름이 말을 아꼈다",
        "텅 빈 마음 위에 구름이 살짝 걸려 있었다",
        "조용한 멜로디처럼 흘러가는 하늘"
    ],
    "scattered clouds": [
        "마음이 복잡할 땐 이런 구름이 딱이야",
        "하늘도 나처럼 갈피를 못 잡는 중",
        "구름이 흐르다가 살짝 멈춘 것 같았어",
        "지나간 감정들이 흩어져 있는 하늘",
        "선명하지 않아 더 오래 바라보게 되는 하늘"
    ],
    "broken clouds": [
        "누가 구름에 빈틈을 내버린 걸까, 나처럼",
        "마치 오래된 기억처럼 퍼져 있는 구름",
        "빛과 그림자가 섞여 있는 하늘",
        "쏟아질 듯 말 듯, 말 걸고 싶은 하늘",
        "무언가 시작되기 직전의 공기"
    ],
    "overcast clouds": [
        "세상이 회색이면 나도 잠깐 쉰다",
        "하늘도 말을 아끼는 날이 있지",
        "구름이 내 머릿속처럼 무거워 보이던 오후",
        "빛이 멀어진 대신 소리에 더 집중하게 된다",
        "이런 날은 피아노 건반도 무거워져"
    ],
    "light rain": [
        "빗방울이 기타줄 위에서 춤추는 것 같았어",
        "조용한 비가 어제의 감정을 꺼내놓는다",
        "촉촉한 공기, 눅눅한 마음, 괜찮아 오늘은",
        "비가 오는 소리보다 내 발소리가 더 크다",
        "빗속에선 울어도 티 안 나서 좋지"
    ],
    "moderate rain": [
        "이 정도 비면 혼자 걷기 딱 좋아",
        "비가 길고 깊게 생각을 끌어낸다",
        "적당한 슬픔엔 적당한 비가 어울려",
        "창문 너머로 뿌옇게 번지는 거리",
        "이어폰보다 비소리가 더 가까워"
    ],
    "heavy rain": [
        "비가 쏟아지는 만큼 감정도 흘려버리고 싶었다",
        "세상이 울고 있는 것 같아 위로해주고 싶었어",
        "쏟아지는 소리에 마음이 잠겨버렸어",
        "비명처럼 쏟아지는 하늘, 그러나 난 잠잠했지",
        "아무도 없는 골목, 비와 나만 남았어"
    ],
    "snow": [
        "세상이 느려져서 좋았어, 눈 덕분에",
        "하얀 게 내려오니까 어제랑은 다른 느낌이었어",
        "눈이 이불처럼 도시를 덮고 있었어",
        "걸을 때마다 소리가 사라지는 느낌이 좋다",
        "눈 내릴 때만 세상이 조금 따뜻해져"
    ],
    "mist": [
        "안개 속은 누가 날 못 알아봐서 좋다",
        "희뿌연 건 가끔 감정을 숨기기에 딱이야",
        "어디쯤인지 모를 풍경이 날 안심시켰어",
        "안개는 마음이 멍한 상태 같기도 해",
        "소리가 더 가까이 들리는 날씨"
    ]
}


In [7]:
temp_mapping = {
    "cold": [
        "숨 쉴 때마다 입김이 노래처럼 흘러나와",
        "손끝이 얼어붙는 계절, 생각은 더 또렷해져",
        "차가운 공기가 나를 더 선명하게 만든다",
        "따뜻한 말 한 마디가 더 그리운 온도",
        "이 날씨엔 커피도 말이 많아져"
    ],
    "cool": [
        "서늘한 바람이 생각을 정리해준다",
        "코끝이 시린 게 왠지 좋았다",
        "이런 날엔 잔잔한 멜로디가 잘 어울려",
        "바람이 살짝 비껴가서 다행이었어",
        "얇은 니트 하나로도 충분한 오후"
    ],
    "mild": [
        "딱 지금 이 온도가 감정 쓰기에 좋아",
        "뭔가를 시작하고 싶어지는 공기",
        "가볍게 걷기 좋은 날씨, 나랑 딱 맞는",
        "아무 이유 없이 기분 좋은 날 있잖아, 오늘이야",
        "손에 닿는 공기가 말랑해서 좋았어"
    ],
    "warm": [
        "따뜻한 햇살에 어깨가 녹아내릴 것 같았어",
        "긴팔은 과하고 반팔은 아쉬운 그런 날",
        "햇빛이 내 마음까지 덥혀줬어",
        "익숙하고 편안한 공기, 마치 네 목소리",
        "지금 이 온도, 그대로 노래에 담고 싶다"
    ],
    "hot": [
        "더워서 생각까지 눅눅해졌어",
        "숨 쉬는 것도 귀찮은 날이지만, 네 생각은 선명해",
        "여름 냄새가 옷에 붙은 채로 집에 돌아왔다",
        "땀이 흐르는데 이상하게 기분은 괜찮더라",
        "오늘은 에어컨보다 코드 진행이 시원했어"
    ]
}


In [8]:
time_mapping = {
    "dawn": [
        "어둠이 아직 깔린 새벽, 가사 쓰기엔 완벽한 시간",
        "세상이 조용할수록 머릿속은 시끄러워져",
        "기타 줄 하나 튕기면 공기가 다 울려",
        "나만 깨어 있는 느낌, 사실 그게 좋아",
        "새벽 공기엔 진짜 말들이 섞여 있어"
    ],
    "morning": [
        "햇살이 창틀을 타고 노래를 시작해",
        "따뜻한 물 한 잔과 기타, 나의 아침 루틴",
        "오늘은 무슨 이야기를 꺼낼까, 고민이 시작되는 시간",
        "창밖 소음이 멜로디처럼 들릴 때가 있어",
        "눈 뜨자마자 코드를 하나 잡았다"
    ],
    "afternoon": [
        "햇빛이 벽에 물드는 오후, 멜로디도 물들었다",
        "지나가는 바람이 기타 소리를 훔쳐갔다",
        "감정이 조금 흐릿해질 때쯤, 노래가 나왔다",
        "오후는 잠깐의 멍이 필요해",
        "창밖은 바쁘지만 난 기타와 한가해"
    ],
    "evening": [
        "노을이 창문에 닿을 때마다 생각나는 목소리",
        "조명이 켜지는 순간부터 감정도 달라져",
        "코드 하나만 바꿔도 분위기가 완전히 달라",
        "노을빛이 쓸쓸하게 방 안으로 들어왔다",
        "지금의 공기, 가사에 꼭 담아야겠다"
    ],
    "night": [
        "밤은 항상 이야기를 만들고 있어",
        "조용한 방 안, 이어폰과 나만 있는 시간",
        "별빛보다 기타가 더 반짝였다",
        "멜로디가 베개처럼 마음을 감쌌다",
        "밤은 창작자에게 제일 친절한 시간"
    ]
}


In [16]:
import requests
from datetime import datetime
import pytz
import random


# 감성 조합 함수
def describe_feeling(weather_desc, feels_like, hour):
    # 날씨
    weather_line = random.choice(weather_mapping.get(weather_desc, [f"{weather_desc}의 분위기"]))

    # 온도
    if feels_like < 0:
        temp_line = random.choice(temp_mapping["cold"])
    elif feels_like < 10:
        temp_line = random.choice(temp_mapping["cool"])
    elif feels_like < 20:
        temp_line = random.choice(temp_mapping["mild"])
    elif feels_like < 28:
        temp_line = random.choice(temp_mapping["warm"])
    else:
        temp_line = random.choice(temp_mapping["hot"])

    # 시간
    if 5 <= hour < 9:
        time_line = random.choice(time_mapping["morning"])
    elif 9 <= hour < 12:
        time_line = random.choice(time_mapping["morning"])
    elif 12 <= hour < 17:
        time_line = random.choice(time_mapping["afternoon"])
    elif 17 <= hour < 20:
        time_line = random.choice(time_mapping["evening"])
    elif 20 <= hour < 24:
        time_line = random.choice(time_mapping["night"])
    else:
        time_line = random.choice(time_mapping["dawn"])

    return f"{time_line}\n{weather_line}\n{temp_line}"

# 날씨 API 요청
def get_weather_story(city_name, api_key):
    url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": api_key,
        "units": "metric",
        "lang": "en"
    }
    res = requests.get(url, params=params)
    if res.status_code != 200:
        return f"API 오류: {res.status_code}"

    data = res.json()

    # 한국 시간 변환
    utc_time = datetime.utcfromtimestamp(data['dt'])
    kst = pytz.timezone('Asia/Seoul')
    local_time = utc_time.replace(tzinfo=pytz.utc).astimezone(kst)
    hour = local_time.hour

    # 데이터 추출
    weather_desc = data["weather"][0]["description"]
    feels_like = data["main"]["feels_like"]

    # 감성 스토리 구성
    story = f"""{local_time.strftime('%Y년 %m월 %d일')} {city_name}의 날씨 이야기\n\n{describe_feeling(weather_desc, feels_like, hour)}"""

    return story

# 메인 함수
if __name__ == "__main__":
    api_key = ""  # 본인 API 키로 대체
    city = "Seoul"
    story = get_weather_story(city, api_key)
    print("오늘의 날씨 스토리")
    print(story)


오늘의 날씨 스토리
2025년 05월 02일 Seoul의 날씨 이야기

창밖은 바쁘지만 난 기타와 한가해
맑은 하늘이 오늘만큼은 나를 응원하는 것 같아
아무 이유 없이 기분 좋은 날 있잖아, 오늘이야


In [ ]:
[날씨API 정보] → [정규화]
         ↓
[날씨/온도/시간대 매핑]
         ↓
[랜덤 감성 문장 선택]
         ↓
[문장 조합 및 출력]